[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# A Small Pipeline &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds everything the notebook set up: the archive, and the five stage
functions. Run it first.


In [1]:

import csv
import io
import json
import os
import shutil
import tempfile
import zipfile
from collections import Counter
from pathlib import Path

scratch = Path("scratch")
if scratch.exists():
    shutil.rmtree(scratch)
scratch.mkdir()

archive = scratch / "readings.zip"
SOURCES = {
    "2026/north.csv": ("utf-8",
        "date,station,celsius\n2026-03-01,Troms\u00f8,-4.1\n2026-03-02,Bod\u00f8,-2.6\n"),
    "2026/south.csv": ("latin-1",
        "date,station,celsius\n2026-03-01,M\u00e1laga,18.9\n2026-03-02,M\u00e1laga,19.4\n"),
    "2026/east.csv": ("utf-8",
        "date,station,celsius\n2026-03-01,Krak\u00f3w,7.2\n2026-03-02,Krak\u00f3w\n"
        "2026-03-03,Krak\u00f3w,warm\n"),
    "2026/west.csv": ("utf-8-sig",
        "date,station,celsius\n2026-03-01,Galway,11.5\n"),
    "notes.txt": ("utf-8", "March readings, collected by four teams.\n"),
}


def build(path, sources):
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for name, (encoding, text) in sources.items():
            z.writestr(name, text.encode(encoding))
    return path


def manifest(archive_path):
    entries = []
    with zipfile.ZipFile(archive_path) as z:
        for info in sorted(z.infolist(), key=lambda i: i.filename):
            if info.is_dir() or not info.filename.lower().endswith(".csv"):
                continue
            entries.append({"name": info.filename, "bytes": info.file_size,
                            "stored": info.compress_size})
    return entries


ATTEMPTS = ("utf-8-sig", "latin-1")
REQUIRED = ("date", "station", "celsius")


def decode(raw):
    for encoding in ATTEMPTS:
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    raise ValueError(f"none of {ATTEMPTS} could decode these bytes")


def clean_rows(text, source):
    kept, dropped = [], []
    for line_no, row in enumerate(csv.DictReader(io.StringIO(text)), start=2):
        missing = [field for field in REQUIRED if not row.get(field)]
        if missing:
            dropped.append({"source": source, "line": line_no,
                            "reason": f"missing {missing[0]}"})
            continue
        try:
            celsius = float(row["celsius"])
        except ValueError:
            dropped.append({"source": source, "line": line_no,
                            "reason": "celsius is not a number"})
            continue
        kept.append({"date": row["date"], "station": row["station"],
                     "celsius": celsius, "source": source})
    return kept, dropped


def summarize(kept, dropped, entries):
    per_station = {}
    for row in kept:
        per_station.setdefault(row["station"], []).append(row["celsius"])
    return {"files_read": len(entries), "rows_read": len(kept) + len(dropped),
            "rows_kept": len(kept), "rows_dropped": len(dropped),
            "reasons": dict(Counter(row["reason"] for row in dropped)),
            "encodings": dict(Counter(entry["encoding"] for entry in entries)),
            "stations": {name: {"n": len(values),
                                "mean_celsius": round(sum(values) / len(values), 2)}
                         for name, values in sorted(per_station.items())}}


def to_csv_text(fieldnames, rows):
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue()


def write_atomic(target, text):
    handle, temporary = tempfile.mkstemp(dir=target.parent, suffix=".part")
    temporary = Path(temporary)
    try:
        with open(handle, "w", encoding="utf-8", newline="") as f:
            f.write(text)
        os.replace(temporary, target)
    finally:
        temporary.unlink(missing_ok=True)


def run(archive_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    entries, kept, dropped = manifest(archive_path), [], []
    with zipfile.ZipFile(archive_path) as z:
        for entry in entries:
            text, entry["encoding"] = decode(z.read(entry["name"]))
            rows_kept, rows_dropped = clean_rows(text, entry["name"])
            kept += rows_kept
            dropped += rows_dropped
    summary = summarize(kept, dropped, entries)
    kept.sort(key=lambda row: (row["source"], row["date"]))
    write_atomic(out_dir / "clean.csv",
                 to_csv_text(["date", "station", "celsius", "source"], kept))
    write_atomic(out_dir / "dropped.csv",
                 to_csv_text(["source", "line", "reason"], dropped))
    write_atomic(out_dir / "report.json",
                 json.dumps(summary, indent=2, ensure_ascii=False))
    return summary


build(archive, SOURCES)
print("ready:", archive.name)


ready: readings.zip


**1.** A fifth team's file, then run the pipeline against the larger archive.


In [2]:

bigger = dict(SOURCES)
bigger["2026/central.csv"] = ("utf-8",
    "date,station,celsius\n2026-03-01,Brno,9.4\n2026-03-02,Brno,10.1\n")

build(scratch / "readings-five.zip", bigger)
report = run(scratch / "readings-five.zip", scratch / "out-five")

print("files_read:", report["files_read"])
print("rows_kept: ", report["rows_kept"])


files_read: 5
rows_kept:  8


Two more files went in (`central.csv` is data, and the manifest counts only CSV members), so
`files_read` goes from four to five and `rows_kept` from six to eight. Nothing else changed,
which is what a stage boundary buys you.


**2.** A range check, as a third rejection reason.


In [3]:

def clean_rows_checked(text, source):
    kept, dropped = [], []
    for line_no, row in enumerate(csv.DictReader(io.StringIO(text)), start=2):
        missing = [field for field in REQUIRED if not row.get(field)]
        if missing:
            dropped.append({"source": source, "line": line_no,
                            "reason": f"missing {missing[0]}"})
            continue
        try:
            celsius = float(row["celsius"])
        except ValueError:
            dropped.append({"source": source, "line": line_no,
                            "reason": "celsius is not a number"})
            continue
        if not -90 <= celsius <= 60:
            dropped.append({"source": source, "line": line_no,
                            "reason": "celsius out of range"})
            continue
        kept.append({"date": row["date"], "station": row["station"],
                     "celsius": celsius, "source": source})
    return kept, dropped


probe = "date,station,celsius\n2026-03-01,Brno,999\n2026-03-02,Brno,9.4\n"
kept, dropped = clean_rows_checked(probe, "probe.csv")

print("kept:   ", kept)
print("dropped:", dropped)


kept:    [{'date': '2026-03-02', 'station': 'Brno', 'celsius': 9.4, 'source': 'probe.csv'}]
dropped: [{'source': 'probe.csv', 'line': 2, 'reason': 'celsius out of range'}]


The range check goes after the `float`, because it needs a number to compare. Each rejection
still records the file and the line, so the new reason is as findable as the other two.


**3.** Compression ratio per member.


In [4]:

def manifest_with_ratio(archive_path):
    entries = manifest(archive_path)
    for entry in entries:
        entry["compression"] = round(entry["stored"] / entry["bytes"], 2)
    return entries


for entry in manifest_with_ratio(archive):
    print(f"  {entry['name']:<16} {entry['compression']}")


  2026/east.csv    0.69
  2026/north.csv   0.88
  2026/south.csv   0.81
  2026/west.csv    1.04


`2026/west.csv` comes out above 1.0, meaning it grew. Deflate writes a small header per member,
and on a forty-seven byte file that header costs more than the compression saves. Ratios like
this are only meaningful on files large enough for the overhead to disappear.


**4.** The attempt order reversed.


In [5]:

def decode_reversed(raw):
    for encoding in reversed(ATTEMPTS):
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue


with zipfile.ZipFile(archive) as z:
    for entry in manifest(archive):
        text, used = decode_reversed(z.read(entry["name"]))
        lines = text.splitlines()
        print(f"  {entry['name']:<16} {used:<10} {lines[0]:<24} {lines[1]}")

# Every file now decodes as latin-1, because latin-1 maps all 256 byte values to
# characters and so can never raise UnicodeDecodeError. Two lines are wrong:
# east.csv reads Krakow with a mangled o, north.csv reads Tromso the same way,
# and west.csv keeps its byte order mark as three visible characters in the
# header, which would make its first column name unusable.


  2026/east.csv    latin-1    date,station,celsius     2026-03-01,KrakÃ³w,7.2
  2026/north.csv   latin-1    date,station,celsius     2026-03-01,TromsÃ¸,-4.1
  2026/south.csv   latin-1    date,station,celsius     2026-03-01,Málaga,18.9
  2026/west.csv    latin-1    ï»¿date,station,celsius  2026-03-01,Galway,11.5


The header damage on `2026/west.csv` is the one that would bite hardest. `DictReader` takes the
first line as the field names, so the date column would be called something other than `date`,
`row.get("date")` would return `None`, and every row in that file would be dropped for a missing
date. A wrong encoding does not always look like wrong text; sometimes it looks like missing data.


**5.** The arithmetic check.


In [6]:

def check(summary):
    return summary["rows_kept"] + summary["rows_dropped"] == summary["rows_read"]


report = run(archive, scratch / "out-check")
print("as reported: ", check(report))

lost_one = dict(report, rows_kept=report["rows_kept"] - 1)
print("one row lost:", check(lost_one))


as reported:  True
one row lost: False


This is worth running at the end of a real pipeline and raising on. It cannot tell you the numbers
are right, only that they are consistent, and inconsistent numbers mean a row went somewhere
without being counted.


**6.** Refuse to overwrite unless told to.


In [7]:

def run_guarded(archive_path, out_dir, overwrite=False):
    target = out_dir / "clean.csv"
    if target.exists() and not overwrite:
        raise FileExistsError(f"{target} already exists; pass overwrite=True to replace it")
    return run(archive_path, out_dir)


try:
    run_guarded(archive, scratch / "out-check")
except FileExistsError as error:
    print("refused: ", error)

report = run_guarded(archive, scratch / "out-check", overwrite=True)
print("replaced:", report["rows_kept"], "rows kept")


refused:  scratch/out-check/clean.csv already exists; pass overwrite=True to replace it
replaced: 6 rows kept


The check happens before the archive is opened. There is no point reading five files and cleaning
eight rows to then discover there is nowhere to put them.


Cleaning up.


In [8]:

shutil.rmtree(scratch)

print("cleaned up:", not scratch.exists())


cleaned up: True


---

&#8592; **Back to:** [A Small Pipeline](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/10-a-small-pipeline.ipynb)  &nbsp;&middot;&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
